# Phase Zero Cyber Muscle — Non-Traditional Sovereign QLoRA Trainer

Fine-tunes the **Tier 1 Hybrid Cyber Muscle** on **Google Colab Free Tier (Tesla T4 16GB VRAM)**.
- **Base Model**: `unsloth/DeepSeek-R1-Distill-Qwen-14B-bnb-4bit`
- **Dataset**: 2,008 curated pairs embedded inline (Phase Zero 4-Pillar v1 + Asymmetric Link Telemetry + KEV/LOLBAS + Neutral Core)
- **Native Nature**: Protection-first, Deceptive Posture, Invisible Wire-Silent Execution, Stealth Non-Traditional Architecture
- **Output**: Compact PEFT adapter bundle `cyber_muscle_adapter.zip` (auto-downloaded in ~10s, converted to `lora_cyber_muscle.gguf` locally)

**How to run**: `Runtime` -> `Run all` (`Ctrl+F9`).

In [ ]:
# Step 1: Install Unsloth & QLoRA Runtime (~45s)
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
# Step 2: Unpack 2,008 Curated Cyber Muscle Training Pairs
import gzip
import base64

CYBER_B64_DATA = "..."
raw_jsonl = gzip.decompress(base64.b64decode(CYBER_B64_DATA)).decode("utf-8")
with open("cyber_muscle_dataset.jsonl", "w", encoding="utf-8") as f:
    f.write(raw_jsonl)

line_count = len(raw_jsonl.strip().splitlines())
print(f"Unpacked {line_count:,} Cyber Muscle pairs ({len(raw_jsonl):,} bytes).")

In [ ]:
# Step 3: Load DeepSeek-R1-Distill-Qwen-14B in 4-bit & Attach LoRA Projections
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/DeepSeek-R1-Distill-Qwen-14B-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("14B model ready with LoRA targets (r=16, alpha=32).")

In [ ]:
# Step 4: Format Mixed ShareGPT & Prompt/Chosen Pairs + Train 200 Steps (~15 min on Tesla T4)
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

dataset = load_dataset("json", data_files="cyber_muscle_dataset.jsonl", split="train")

def format_prompts(batch):
    num_examples = len(batch[next(iter(batch))])
    texts = []
    has_tpl = hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None
    convos_col = batch.get("conversations", [None] * num_examples)
    prompt_col = batch.get("prompt", [None] * num_examples)
    chosen_col = batch.get("chosen", [None] * num_examples)

    for i in range(num_examples):
        convo = convos_col[i]
        prompt = prompt_col[i]
        chosen = chosen_col[i]
        if convo and isinstance(convo, list):
            msgs = [{"role": ("user" if t.get("from") in ("human", "user") else "assistant"), "content": t.get("value", "")} for t in convo if isinstance(t, dict)]
        elif prompt and chosen:
            msgs = [{"role": "user", "content": str(prompt)}, {"role": "assistant", "content": str(chosen)}]
        else:
            texts.append("")
            continue
        if has_tpl:
            texts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
        else:
            texts.append("\n".join(f"<{m['role']}>\n{m['content']}" for m in msgs))
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True, remove_columns=dataset.column_names)
dataset = dataset.filter(lambda ex: len(ex["text"].strip()) > 0)
print(f"Formatted {len(dataset):,} valid non-empty training examples.")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=12,
        max_steps=200,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

stats = trainer.train()
print(f"Training complete! Final Loss: {stats.training_loss:.4f}")

In [ ]:
# Step 5: Export Compact PEFT LoRA Bundle (~255MB SafeTensors) & Download Immediately
# Fast & safe: avoids 28GB full-base merge OOM; converted to GGUF locally in 3s by colab_automation_sidecar.py
import shutil
from google.colab import files

model.save_pretrained("cyber_muscle_adapter")
tokenizer.save_pretrained("cyber_muscle_adapter")
shutil.make_archive("cyber_muscle_adapter", "zip", "cyber_muscle_adapter")
print("Exported cyber_muscle_adapter.zip cleanly! Initiating download...")
files.download("cyber_muscle_adapter.zip")